# Getting basic dataset descriptors

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm

import warnings;warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'updated-data/obs'
# ENGLISH_DATA_PATH = 'english/data/ckpts/rosen'
# XLING_DATA_PATH = 'xling/data/ckpts/rosen'


In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
fs = [f for f in grab_all_files(DATA_PATH) if ('yid' not in f)]
len(fs)

5564

## Processing individual datasets

In [5]:
data, needs_avar = [], []

In [6]:
for f in tqdm(fs):
    table = pq.ParquetFile(f)
    df = table.read(columns=['corpus', 'conversation_id', 'x_speaker']).to_pandas()

    if 'AVAR' not in table.schema.names:
        needs_avar += [f]

    for corpus in df['corpus'].unique():
        sub = df.loc[df['corpus'].isin([corpus])]
        lang = 'eng'
        if 'call' in corpus:
            lang = corpus.split('-')[-1].split('_')[0]
        data += [
            {
                'xling': 'xling-' in f,
                'corpus': corpus,
                # 'lang': lang,
                'conversations': sub['conversation_id'].nunique(),
                'speakers': sub['x_speaker'].nunique(),
                'comparisons': len(sub),
            }
        ]

data = pd.DataFrame(data)

  0%|          | 0/5564 [00:00<?, ?it/s]

In [7]:
needs_avar

[]

In [19]:
# data['corpus'] = ['CANDOR' if ('CANDOR' in it) else it for it in data['corpus']]
# data['corpus'] = ['MICASE' if ('MICASE' in it) else it for it in data['corpus']]

In [8]:
data = data.groupby(by=['xling', 'corpus']).agg('sum').reset_index()

In [9]:
data.head(n=30)

,xling,corpus,conversations,speakers,comparisons
0,False,CABNC,1997,6126,43164420
1,False,CANDOR,1653,3306,89745440
2,False,CORAAL,271,588,56663658
3,False,DISPEL,19,38,485633
4,False,Frederiksen,2,4,3709
5,False,GCSAusE,40,106,324155
6,False,Graesser,3,6,52652
7,False,MICASE,152,1756,41520798
8,False,SBCSAE,60,340,12398227
9,False,SCoSE,32,87,2770223


In [10]:
data['conversations'].sum(), data['speakers'].sum(), data['comparisons'].sum()

(np.int64(5564), np.int64(15514), np.int64(354803021))

English data characteristics

In [11]:
data['conversations'].loc[~data['xling']].sum(), data['speakers'].loc[~data['xling']].sum(), data['comparisons'].loc[~data['xling']].sum()

(np.int64(4518), np.int64(12986), np.int64(269021928))

In [12]:
# data[[col for col in list(data) if col not in ['xling']]].loc[~data['xling']].to_csv('english-data-stats.csv', index=False, encoding='utf8')
data[[col for col in list(data) if col not in ['xling']]].loc[~data['xling']].to_csv('/Users/zacharyrosen/Desktop/english-data-stats.csv', index=False, encoding='utf8')

Xling data characteristics

In [13]:
sel = data['xling'] | np.array([('call' in c) for c in data['corpus']])
data['conversations'].loc[sel].sum(), data['speakers'].loc[sel].sum(), data['comparisons'].loc[sel].sum()

(np.int64(1262), np.int64(2993), np.int64(103088769))

In [14]:
data.loc[sel].head(n=15)

,xling,corpus,conversations,speakers,comparisons
10,False,callfriend-eng_n,30,68,5420692
11,False,callfriend-eng_s,10,25,1442843
12,False,callhome,176,372,10444141
15,True,callfriend-fra-q,60,136,11641191
16,True,callfriend-ko,100,212,9170611
17,True,callfriend-spa-c,53,53,3437758
18,True,callfriend-spa-s,124,303,28655745
19,True,callfriend-zho,60,60,5244680
20,True,callhome-deu,120,260,5473047
21,True,callhome-spa,140,356,3968514


In [15]:
# data[[col for col in list(data) if col not in ['xling']]].loc[data['xling']].to_csv('xling-data-stats.csv', index=False, encoding='utf8')
data[[col for col in list(data) if col not in ['xling']]].loc[sel].to_csv('/Users/zacharyrosen/Desktop/xling-data-stats.csv', index=False, encoding='utf8')